## 4.3 앙상블 학습 개요

### Voting Classifier

In [7]:
# 데이터 처리를 위한 pandas 라이브러리 불러오기
# pandas는 표 형태의 데이터를 DataFrame 형태로 다루기 위해 사용한다.
import pandas as pd

# 여러 개의 분류 모델을 결합해서 사용하는 VotingClassifier 불러오기
from sklearn.ensemble import VotingClassifier

# 개별 분류 모델로 사용할 로지스틱 회귀 모델 불러오기
from sklearn.linear_model import LogisticRegression

# 개별 분류 모델로 사용할 K-최근접 이웃 모델 불러오기
from sklearn.neighbors import KNeighborsClassifier

# 예제 데이터셋인 유방암 데이터셋 불러오기
from sklearn.datasets import load_breast_cancer

# 데이터를 학습용 데이터와 테스트용 데이터로 나누기 위한 함수
from sklearn.model_selection import train_test_split

# 모델의 정확도를 계산하기 위한 함수
from sklearn.metrics import accuracy_score

# 사이킷런에서 제공하는 유방암 데이터셋을 로드한다.
cancer = load_breast_cancer()

# cancer.data는 피처 데이터이고, cancer.feature_names는 각 피처의 이름이다.
# 이를 pandas DataFrame으로 변환하여 데이터를 표 형태로 확인할 수 있게 한다.
data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# 데이터의 앞 3개 행을 출력하여 데이터 구조를 확인한다.
data_df.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [8]:
# 개별 모델로 로지스틱 회귀 모델을 생성한다.
# solver='liblinear'는 작은 데이터셋이나 이진 분류 문제에서 안정적으로 사용할 수 있는 최적화 방법이다.
lr_clf = LogisticRegression(solver='liblinear')

# 개별 모델로 KNN 분류 모델을 생성한다.
# n_neighbors=8은 예측할 때 가장 가까운 8개의 이웃 데이터를 참고하겠다는 의미이다.
knn_clf = KNeighborsClassifier(n_neighbors=8)

# 로지스틱 회귀 모델과 KNN 모델을 결합한 VotingClassifier를 생성한다.
# estimators에는 결합할 개별 모델들을 이름과 함께 넣는다.
# voting='soft'는 각 모델의 예측 확률을 평균내어 최종 클래스를 결정하는 방식이다.
vo_clf = VotingClassifier(
    estimators=[('LR', lr_clf), ('KNN', knn_clf)],
    voting='soft'
)

# 전체 데이터를 학습 데이터와 테스트 데이터로 분리한다.
# test_size=0.2는 전체 데이터 중 20%를 테스트 데이터로 사용한다는 의미이다.
# random_state=156은 실행할 때마다 같은 방식으로 데이터가 나뉘도록 고정하는 값이다.
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data,
    cancer.target,
    test_size=0.2,
    random_state=156
)

# VotingClassifier 모델을 학습 데이터로 학습시킨다.
vo_clf.fit(X_train, y_train)

# 학습된 VotingClassifier 모델로 테스트 데이터를 예측한다.
pred = vo_clf.predict(X_test)

# 실제 정답 y_test와 예측값 pred를 비교하여 정확도를 출력한다.
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

# VotingClassifier에 사용된 개별 모델들의 성능도 따로 비교하기 위해 리스트로 묶는다.
classifiers = [lr_clf, knn_clf]

# 각 개별 모델을 하나씩 학습하고 예측 성능을 확인한다.
for classifier in classifiers:
    # 개별 모델 학습
    classifier.fit(X_train, y_train)

    # 개별 모델로 테스트 데이터 예측
    pred = classifier.predict(X_test)

    # 모델 클래스 이름을 가져온다.
    class_name = classifier.__class__.__name__

    # 각 개별 모델의 정확도를 출력한다.
    print('{0} 정확도: {1:.4f}'.format(class_name, accuracy_score(y_test, pred)))

Voting 분류기 정확도: 0.9561
LogisticRegression 정확도: 0.9474
KNeighborsClassifier 정확도: 0.9386


## 4.4 Random Forest

In [9]:
# 중복된 피처 이름을 수정하기 위한 함수 정의
# Human Activity Recognition 데이터셋에는 동일한 이름의 피처가 존재할 수 있다.
# pandas DataFrame에서는 중복 컬럼명이 있으면 분석 과정에서 문제가 생길 수 있으므로
# 중복된 피처명 뒤에 _1, _2와 같은 번호를 붙여 고유한 컬럼명으로 바꿔준다.
def get_new_feature_name_df(old_feature_name_df):

    # 같은 column_name이 몇 번째로 중복되는지 누적 번호를 계산한다.
    # 처음 등장한 값은 0, 두 번째 등장한 값은 1, 세 번째 등장한 값은 2가 된다.
    feature_dup_df = pd.DataFrame(
        data=old_feature_name_df.groupby('column_name').cumcount(),
        columns=['dup_cnt']
    )

    # 기존 인덱스를 컬럼으로 변환하여 병합할 수 있는 형태로 만든다.
    feature_dup_df = feature_dup_df.reset_index()

    # 기존 피처 이름 DataFrame과 중복 횟수 정보를 병합한다.
    new_feature_name_df = pd.merge(
        old_feature_name_df.reset_index(),
        feature_dup_df,
        how='outer'
    )

    # 중복 횟수 dup_cnt가 0보다 크면 기존 피처명 뒤에 _번호를 붙인다.
    # 예를 들어 같은 이름이 두 번째로 등장하면 feature_1 형태로 바꾼다.
    # 처음 등장한 피처는 원래 이름을 그대로 사용한다.
    new_feature_name_df['column_name'] = new_feature_name_df[
        ['column_name', 'dup_cnt']
    ].apply(
        lambda x: x[0] + '_' + str(x[1]) if x[1] > 0 else x[0],
        axis=1
    )

    # 불필요한 index 컬럼을 제거한다.
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)

    # 중복 문제가 해결된 새로운 피처명 DataFrame을 반환한다.
    return new_feature_name_df

In [10]:
# 데이터 처리를 위해 pandas 라이브러리를 불러온다.
import pandas as pd

# Human Activity Recognition 데이터셋을 불러오는 함수 정의
# 이 함수는 학습용 피처, 테스트용 피처, 학습용 레이블, 테스트용 레이블을 반환한다.
def get_human_dataset():

    # features.txt 파일에는 각 피처의 번호와 이름이 저장되어 있다.
    # 데이터가 공백으로 구분되어 있으므로 sep='\s+'를 사용한다.
    # header=None은 파일에 별도의 헤더가 없다는 의미이다.
    # names를 이용해 컬럼명을 column_index와 column_name으로 지정한다.
    feature_name_df = pd.read_csv(
        './human_activity/features.txt',
        sep='\s+',
        header=None,
        names=['column_index', 'column_name']
    )

    # 중복된 피처명을 수정하는 함수를 적용하여 새로운 피처명 DataFrame을 만든다.
    new_feature_name_df = get_new_feature_name_df(feature_name_df)

    # DataFrame에 컬럼명으로 넣기 위해 피처 이름만 리스트 형태로 추출한다.
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()

    # 학습용 피처 데이터를 불러온다.
    # X_train.txt도 공백 기준으로 값이 구분되어 있으므로 sep='\s+'를 사용한다.
    # 컬럼명은 위에서 만든 feature_name 리스트를 적용한다.
    X_train = pd.read_csv(
        './human_activity/train/X_train.txt',
        sep='\s+',
        names=feature_name
    )

    # 테스트용 피처 데이터를 불러온다.
    # 학습 데이터와 동일한 피처명을 사용한다.
    X_test = pd.read_csv(
        './human_activity/test/X_test.txt',
        sep='\s+',
        names=feature_name
    )

    # 학습용 레이블 데이터를 불러온다.
    # 각 데이터가 어떤 동작 클래스에 해당하는지 저장되어 있다.
    # 컬럼명은 action으로 지정한다.
    y_train = pd.read_csv(
        './human_activity/train/y_train.txt',
        sep='\s+',
        header=None,
        names=['action']
    )

    # 테스트용 레이블 데이터를 불러온다.
    # 테스트 데이터의 실제 정답 클래스가 저장되어 있다.
    y_test = pd.read_csv(
        './human_activity/test/y_test.txt',
        sep='\s+',
        header=None,
        names=['action']
    )

    # 학습 피처, 테스트 피처, 학습 레이블, 테스트 레이블을 반환한다.
    return X_train, X_test, y_train, y_test


# 위에서 정의한 함수를 실행하여 데이터를 불러온다.
X_train, X_test, y_train, y_test = get_human_dataset()

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
<>:47: SyntaxWarning: invalid escape sequence '\s'
<>:56: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
<>:47: SyntaxWarning: invalid escape sequence '\s'
<>:56: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_17175/716383048.py:14: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',
/tmp/ipykernel_17175/716383048.py:30: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',
/tmp/ipykernel_17175/716383048.py:38: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',
/tmp/ipykernel_17175/716383048.py:47: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',
/tmp/ipykernel_17175/716383048.py:56: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',


FileNotFoundError: [Errno 2] No such file or directory: './human_activity/features.txt'

In [ ]:
# 랜덤 포레스트 분류 모델을 사용하기 위해 불러온다.
from sklearn.ensemble import RandomForestClassifier

# 모델의 정확도를 계산하기 위한 함수이다.
from sklearn.metrics import accuracy_score

# 데이터 처리를 위해 pandas를 불러온다.
import pandas as pd

# 경고 메시지를 무시하기 위해 warnings 라이브러리를 불러온다.
import warnings
warnings.filterwarnings('ignore')

# 앞에서 정의한 get_human_dataset() 함수를 이용해 데이터를 불러온다.
# X_train, X_test는 피처 데이터이고, y_train, y_test는 정답 레이블이다.
X_train, X_test, y_train, y_test = get_human_dataset()

# 랜덤 포레스트 분류 모델을 생성한다.
# random_state=0은 결과 재현성을 위해 난수값을 고정하는 역할을 한다.
rf_clf = RandomForestClassifier(random_state=0)

# 랜덤 포레스트 모델을 학습 데이터로 학습시킨다.
rf_clf.fit(X_train, y_train)

# 학습된 모델을 이용해 테스트 데이터의 클래스를 예측한다.
pred = rf_clf.predict(X_test)

# 실제 정답 y_test와 예측값 pred를 비교하여 정확도를 계산한다.
accuracy = accuracy_score(y_test, pred)

# 랜덤 포레스트 모델의 테스트 정확도를 출력한다.
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy))

In [ ]:
# 하이퍼파라미터 튜닝을 위해 GridSearchCV를 불러온다.
# GridSearchCV는 여러 하이퍼파라미터 조합을 비교하여 가장 좋은 조합을 찾아준다.
from sklearn.model_selection import GridSearchCV

# 랜덤 포레스트에서 튜닝할 하이퍼파라미터 후보들을 딕셔너리 형태로 설정한다.
params = {
    # n_estimators는 랜덤 포레스트를 구성하는 결정 트리의 개수이다.
    'n_estimators': [100],

    # max_depth는 각 결정 트리의 최대 깊이이다.
    # 깊이가 깊을수록 복잡한 모델이 되지만 과적합 위험도 커질 수 있다.
    'max_depth': [6, 8, 10, 12],

    # min_samples_leaf는 리프 노드가 되기 위해 필요한 최소 샘플 수이다.
    # 값이 클수록 트리가 덜 복잡해지고 과적합을 줄이는 효과가 있다.
    'min_samples_leaf': [8, 12, 18],

    # min_samples_split은 노드를 분할하기 위해 필요한 최소 샘플 수이다.
    # 값이 클수록 분할이 제한되어 모델이 단순해진다.
    'min_samples_split': [8, 16, 20]
}

# RandomForestClassifier 객체를 생성한다.
# random_state=0은 결과 재현성을 위한 설정이다.
# n_jobs=-1은 가능한 모든 CPU 코어를 사용하여 학습 속도를 높이겠다는 의미이다.
rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)

# GridSearchCV 객체를 생성한다.
# param_grid=params는 위에서 설정한 하이퍼파라미터 조합을 탐색하겠다는 의미이다.
# cv=2는 2-fold 교차검증을 수행한다는 의미이다.
# n_jobs=-1은 가능한 모든 CPU 코어를 사용한다는 의미이다.
grid_cv = GridSearchCV(
    rf_clf,
    param_grid=params,
    cv=2,
    n_jobs=-1
)

# 학습 데이터를 이용하여 모든 하이퍼파라미터 조합을 학습 및 검증한다.
grid_cv.fit(X_train, y_train)

# 가장 성능이 좋았던 하이퍼파라미터 조합을 출력한다.
print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)

# 교차검증에서 얻은 최고 평균 정확도를 출력한다.
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
# GridSearchCV 결과를 참고하여 랜덤 포레스트 모델을 새로 생성한다.
# n_estimators=300은 300개의 결정 트리를 사용한다는 의미이다.
# max_depth=10은 각 트리의 최대 깊이를 10으로 제한한다.
# min_samples_leaf=8은 리프 노드에 최소 8개의 샘플이 있어야 한다는 의미이다.
# min_samples_split=8은 노드를 분할하기 위해 최소 8개의 샘플이 필요하다는 의미이다.
# random_state=0은 결과 재현성을 위한 설정이다.
rf_clf1 = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=8,
    min_samples_split=8,
    random_state=0
)

# 설정한 하이퍼파라미터를 가진 랜덤 포레스트 모델을 학습시킨다.
rf_clf1.fit(X_train, y_train)

# 학습된 모델로 테스트 데이터의 클래스를 예측한다.
pred = rf_clf1.predict(X_test)

# 실제 정답과 예측값을 비교하여 테스트 정확도를 출력한다.
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

In [ ]:
# 그래프를 그리기 위해 matplotlib의 pyplot을 불러온다.
import matplotlib.pyplot as plt

# 더 보기 좋은 시각화를 위해 seaborn 라이브러리를 불러온다.
import seaborn as sns

# 주피터 노트북 안에서 그래프가 바로 출력되도록 설정한다.
%matplotlib inline

# 학습된 랜덤 포레스트 모델에서 각 피처의 중요도 값을 가져온다.
# feature_importances_는 각 피처가 예측에 얼마나 기여했는지를 수치로 나타낸다.
ftr_importances_values = rf_clf1.feature_importances_

# 피처 중요도 값을 pandas Series 형태로 변환한다.
# index에는 각 피처 이름을 넣어 어떤 피처가 중요한지 확인할 수 있게 한다.
ftr_importances = pd.Series(
    ftr_importances_values,
    index=X_train.columns
)

# 피처 중요도를 내림차순으로 정렬한 뒤 상위 20개만 선택한다.
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

# 그래프 크기를 설정한다.
plt.figure(figsize=(8, 6))

# 그래프 제목을 설정한다.
plt.title('Feature importances Top 20')

# 상위 20개 피처 중요도를 막대그래프로 시각화한다.
# x축은 중요도 값, y축은 피처 이름이다.
sns.barplot(x=ftr_top20, y=ftr_top20.index)

# 현재 그래프 객체를 fig1에 저장한다.
# 이후 파일로 저장할 때 사용한다.
fig1 = plt.gcf()

# 그래프를 화면에 출력한다.
plt.show()

# 그래프를 다시 그리는 명령이다.
plt.draw()

# 생성한 그래프를 tif 이미지 파일로 저장한다.
# dpi=300은 고해상도로 저장한다는 의미이다.
# bbox_inches='tight'는 그래프 주변의 불필요한 여백을 줄여 저장한다.
fig1.savefig(
    'rf_feature_importances_top20.tif',
    format='tif',
    dpi=300,
    bbox_inches='tight'
)